<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 2 (Martes) — Clasificación 1 — Árboles de decisión y bosques aleatorios**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🏬 Árbol de decisión: ¿la compra fue en tienda física? (Blackwell)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Predecir si una venta se hizo en tienda (`in.store`) según edad, artículos, monto y región. |
| 🔍 Comprensión de los datos | Cargamos `sales.csv` (10 000 registros), revisamos nulos, proporciones y gráficos exploratorios. |
| 🧹 Preparación de los datos | Dividimos entrenamiento/prueba (70/30) y validamos rangos. |
| 🤖 Modelado | Entrenamos un `DecisionTreeClassifier` con `scikit-learn`. |
| 📊 Evaluación | Matriz de confusión, exactitud, sensibilidad, especificidad, ROC y AUC. |
| 🚀 Despliegue y conclusiones | Conclusiones y recomendaciones al negocio. |

> 🎯 **Analogía:** un vendedor que aprende, por edad, cantidad de artículos y monto, si un cliente suele comprar en la tienda o en línea.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 👉 **2** | Martes | **Clasificación 1: árboles de decisión, bosques aleatorios y KNN** |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('sales.csv', 'dia_3')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

# Investigar  los patrones de la rentabilidad de los productos de Whitewell

El algoritmo produce una función que establece una correspondencia entre las entradas y las salidas deseadas del sistema. Un ejemplo de este tipo de algoritmo es el problema de clasificación, donde el sistema de aprendizaje trata de etiquetar (clasificar) una serie de vectores utilizando una entre varias categorías (clases). La base de conocimiento del sistema está formada por ejemplos de etiquetados anteriores.

- Arboles de decision

### Autores
   * Efrén Jiménez

---
## 🎯 Fase 1 · Comprensión del negocio

Predecir si una venta se hizo en tienda (`in.store`) según edad, artículos, monto y región.

📚 Lectura recomendada: [6]

## Análisis del Problema

Whitewell ha sido un exitoso minorista de productos electrónicos durante más de 40 años, con más de 30 tiendas en el sudeste. Hace poco más de un año lanzo un nuestro sitio web de comercio electrónico. Estamos comenzando a acumular datos de transacciones de clientes del sitio y queremos aprovechar estos datos para informar nuestras decisiones sobre actividades relacionadas con el sitio, como marketing en línea, mejoras en el sitio, etc., para continuar maximizando la cantidad de ingresos que generamos de las ventas de comercio electrónico.

- ¿Los clientes en diferentes regiones gastan más por transacción?
- ¿Qué regiones gastan más / menos?
- ¿Existen diferencias en la edad de los clientes entre regiones? Si es así, ¿podemos predecir la edad de un cliente en una región en función de otros datos demográficos?
- Necesitamos investigar la hipótesis: ¿Existe alguna correlación entre la edad de un cliente y si la transacción se realizó en línea o en la tienda? ¿Algún otro factor predice si un cliente comprará en línea o en nuestras tiendas?
- ¿existe una relación entre la cantidad de artículos comprados y la cantidad gastada?

### Librerías

In [ ]:
import warnings;
warnings.filterwarnings('ignore');

#Manejo de datasets
import pandas as pd
#Manejo de datasets
import numpy as np
#Gráficos
import matplotlib.pyplot as plt
#Gráficos
%matplotlib inline
#Gráficos basados en matplotlib
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
#from sklearn.externals.six import StringIO
from IPython.display import Image
from sklearn.tree import export_graphviz
import pydotplus
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos `sales.csv` (10 000 registros), revisamos nulos, proporciones y gráficos exploratorios.

📚 Lectura recomendada: [9] [10]

## Entendimiento de los Datos

El total es de 10000 registro con 5 columnas dentro del set de datos de Blackwell

- in.store : variable de tipo cualitativa nominal
- age      : variable de tipo cuantitativa discreta   
- items    : variable de tipo cuantitativa discreta  
- amount   : variable de tipo cuantitativa continua  
- region   : variable de tipo cualitativa nominal  

In [ ]:
#Cargar los datos
sales = pd.read_csv(r"sample_data/sales.csv")

In [ ]:
#Primeros registros
sales.head()

In [ ]:
#Describir dataset
sales.describe()

In [ ]:
#Información del dataset
sales.info()

In [ ]:
# Numero de registros del dataset
print("Cantidad de registros:",len(sales))
# Cantidad de columnas del dataset
print("Cantidad de columnas:",len(sales.columns))
# Tipos de datos del dataset
print("Tipo de datos:",sales.dtypes.unique())
# Tamaño del daset
print("Filas y columnas:",sales.shape)


In [ ]:
# Chekear valores nulos
print(sales.isnull().any().sum(), ' / ', len(sales.columns))

In [ ]:
#Cuales columnnas estas con valores nulos
sales.isnull().sum()[sales.isnull().sum() > 0]

In [ ]:
# N/A= Valores vacios
#Usar solo si se quiere eliminar los registros que contienen columnas vacías
sales=sales.dropna()

In [ ]:
sales["in.store"].unique()

In [ ]:
sales['in.store'].value_counts(normalize=True)

In [ ]:
count = sales['in.store'].value_counts(normalize=True)
# Crear un diagrama de barras
plt.bar(count.index, count.values)

---
## 🔍 Fase 2 · Comprensión de los datos — gráficos exploratorios

Cargamos `sales.csv` (10 000 registros), revisamos nulos, proporciones y gráficos exploratorios.

#### Gráficos exploratorios

In [ ]:
plt.figure(1 , figsize = (15 , 6))
n = 0
for x in ['age' , 'items' , 'amount']:
    n += 1
    plt.subplot(1 , 3 , n)
    plt.subplots_adjust(hspace =1 , wspace = 1)
    sns.distplot(sales[x] , bins = 20)
    plt.title('Distribucion de {}'.format(x))
plt.show()

In [ ]:
plt.figure(1 , figsize = (15 , 5))
sns.countplot(y = 'in.store' , data = sales)
plt.show()

Corelacciones entre las variables

In [ ]:
plt.figure(1 , figsize = (15 , 7))
n = 0
for x in ['age' , 'items' , 'amount']:
    for y in ['age' , 'items' , 'amount']:
        n += 1
        plt.subplot(3 , 3 , n)
        plt.subplots_adjust(hspace = 0.5 , wspace = 0.5)
        sns.regplot(x = x , y = y , data = sales)
        plt.ylabel(y.split()[0]+' '+y.split()[1] if len(y.split()) > 1 else y )
plt.show()

In [ ]:
sales['tipo_edad'] = np.select(
        [
            sales['age'] < 18,
            (sales['age'] >= 18) & (sales['age'] < 65),
            sales['age'] >= 65,
        ],
        [
            'menor',
            'joven',
            'adulto'
        ],
        default='joven'
    )

In [ ]:
labels = sales.tipo_edad.value_counts().index
colors = ['green','blue','red']
sizes = sales.tipo_edad.value_counts().values
explode = [0] * len(sizes)

# visual
plt.figure(figsize = (7,7))
plt.pie(sizes, explode=explode, labels=labels, colors=colors[:len(sizes)], autopct='%1.1f%%')
plt.title('Personas segun la edad de compra',color = 'blue',fontsize = 15)

In [ ]:
sns.boxplot(x="in.store", y="amount", hue="in.store", data=sales, palette="PRGn")
plt.show()

In [ ]:
sns.swarmplot(x="in.store", y="amount",hue="in.store", data=sales)
plt.show()

In [ ]:
sns.pairplot(sales)
plt.show()

---
## 🧹 Fase 3 · Preparación de los datos

Dividimos entrenamiento/prueba (70/30) y validamos rangos.

📚 Lectura recomendada: [6]

In [ ]:
#Dividir el conjunto de datos en uno de entrenamiento y otro de pruebas:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(sales[["age","items","amount","region"]],sales[["in.store"]], test_size=0.3, random_state=1234)

>Es importante siempre validar los rangos de los conjuntos de datos creados, para evitar caer en extrapolación:

In [ ]:
print("Entrenamiento dimensiones",X_train.shape)
print("Descripción")
X_train.describe

In [ ]:
print("Pruebas dimensiones",X_test.shape)
print("Descripción")
X_train.describe

---
## 🤖 Fase 4 · Modelado

Entrenamos un `DecisionTreeClassifier` con `scikit-learn`.

📚 Lectura recomendada: [3] [7]

Una vez seleccionadas las variables para incluir en el modelo de regresión, se procede a crearlo:

##### Arbol de desición

![Estructura de un árbol de decisión](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_17.png)
*Figura: nodos internos, ramas y hojas. [8]*

In [ ]:
#Algoritmos de scikit-learn
#https://scikit-learn.org/stable/supervised_learning.html

#Generar el modelo de regresión lineal
tree = DecisionTreeClassifier()

In [ ]:
# Entrenamos nuestro modelo
tree.fit(X_train, y_train)

In [ ]:
# Hacemos las predicciones que en definitiva una línea (en este caso, al ser 2D)
y_pred = tree.predict(X_test)

In [ ]:
predict_values=tree.predict_proba(X_test)
predict_values[:, 1]

---
## 📊 Fase 5 · Evaluación

Matriz de confusión, exactitud, sensibilidad, especificidad, ROC y AUC.

📚 Lectura recomendada: [4] [5]

In [ ]:
matrix = confusion_matrix(y_test, tree.predict(X_test))   # etiquetas, no probabilidades
matrix

In [ ]:
auc = roc_auc_score(y_test, predict_values[:, 1])
print('AUC: %.2f' % auc)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, predict_values[:, 1])

![Curvas ROC](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_10.png)
*Figura: ejemplo de curvas ROC y su AUC. [8]*

In [ ]:
plt.plot(fpr, tpr, color='orange', label='ROC')

plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.show()

Calidad del modelo de machine learning

\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}

In [ ]:
(matrix[0, 0] + matrix[1, 1]) / matrix.sum()

\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

In [ ]:
matrix[1, 1] / (matrix[1, 0] + matrix[1, 1])   # sensibilidad = VP / (VP + FN)

\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}

In [ ]:
matrix[0, 0] / (matrix[0, 0] + matrix[0, 1])   # especificidad = VN / (VN + FP)

---
## 🚀 Fase 6 · Despliegue y conclusiones

Conclusiones y recomendaciones al negocio.

📚 Lectura recomendada: [5]

## Resultados
En conclusión, los valores obtenidos por la clasifición verdaderos positivos no son buenos. Por lo cual no se recomienda utilizar este modelo de machine learning.

---
## 🧪 Refuerzo: profundidad del árbol y sobreajuste en datos de ventas

El árbol por defecto crece hasta separar todo el entrenamiento: casi seguro **sobreajusta**. Veamos cómo cambian la exactitud de entrenamiento y de prueba al limitar la profundidad [6].

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

prof = range(1, 16)
tr, te = [], []
for d in prof:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train.values.ravel())
    tr.append(t.score(X_train, y_train)); te.append(t.score(X_test, y_test))
plt.plot(prof, tr, 'o-', label='Entrenamiento'); plt.plot(prof, te, 's-', label='Prueba')
plt.xlabel('Profundidad máxima'); plt.ylabel('Exactitud'); plt.legend(); plt.show()
print('Mejor profundidad en prueba:', list(prof)[te.index(max(te))])

### ✍️ Ejercicios
1. ¿Qué profundidad recomendaría y por qué (no basta mirar solo la prueba)?
2. Use `class_weight='balanced'` y compare el *recall* de la clase minoritaria.

---
## ✅ Conclusiones y siguientes pasos

**Corrección importante:** las fórmulas de sensibilidad y especificidad de la versión anterior calculaban en realidad el valor predictivo negativo y la precisión; ahora usan VP/(VP+FN) y VN/(VN+FP) [5]. Un árbol sin poda sobreajusta: ver la práctica de profundidad arriba.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [11].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.

**[4]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[5]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[6]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[7]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[8]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[9]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[10]** Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021

**[11]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55